In [5]:
import pandas as pd
import gradio as gr
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score # Import des métriques

# --- 1. DATA LOADING AND INITIALIZATION ---

# Use a portable relative path and a FileNotFoundError check.
# The 'studentPerformance.csv' file MUST be in the same directory as this script.
try:
    dt = pd.read_csv("studentPerformance.csv")
except FileNotFoundError:
    print("--- ERROR: DATA FILE NOT FOUND ---")
    print("The file 'studentPerformance.csv' must be placed in the same directory as this Python script.")
    sys.exit(1)

# Séparer Features et Targets
X = dt[["Study_Hours", "Attendance", "Practice_Tests"]]  # Features
y = dt["Final_Score"]    # Score final (Regression target)
z = dt["Pass_Fail"]      # 0 ou 1 (Classification target)

# Découpage train/test (using the three targets together)
# test_size=0.3 means 30% of data is used for testing
x_train, x_test, y_train, y_test, z_train, z_test = train_test_split(
    X, y, z, test_size=0.3, random_state=2
)

# Entraînement des modèles
# 1. Linear Regression for the continuous score prediction
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)

# 2. Logistic Regression for the binary Pass/Fail prediction
reg_log = LogisticRegression(solver="liblinear", random_state=42)
reg_log.fit(x_train, z_train)


# --- 3. ÉVALUATION DES MODÈLES (Ajout pour le calcul du taux d'erreur) ---

# Prédiction sur l'ensemble de test
y_pred_lr = lr_model.predict(x_test)
z_pred_log = reg_log.predict(x_test)

# 1. Métriques Régression Linéaire (Score)
# Erreur Absolue Moyenne (MAE)
mae = mean_absolute_error(y_test, y_pred_lr)
# Taux d'Erreur Relatif (MAE en pourcentage de la moyenne réelle)
y_mean = y_test.mean()
mae_percent = (mae / y_mean) * 100

# 2. Métriques Régression Logistique (Pass/Fail)
# Précision (Accuracy)
accuracy = accuracy_score(z_test, z_pred_log)


# =============================
# 4. FONCTION DE PRÉDICTION
# =============================
def predict(study_hours, attendance, practice_tests):
    """
    Prédit le score final (cappé entre 0 et 100), la probabilité de réussite, 
    et le résultat Pass/Fail à partir des entrées utilisateur.
    """
    # Créer un DataFrame avec les entrées pour la prédiction
    user_data = pd.DataFrame([[study_hours, attendance, practice_tests]],
                             columns=["Study_Hours", "Attendance", "Practice_Tests"])

    # 1. Prédire score final (Régression Linéaire)
    pred_score_raw = lr_model.predict(user_data)[0]

    # --- CORRECTION: Capping the score between 0 and 100 ---
    # This ensures the output is always within the valid range.
    pred_score = max(0, min(100, pred_score_raw))
    
    # 2. Prédire Pass/Fail (Régression Logistique)
    pred_pass = reg_log.predict(user_data)[0]
    # probabilité classe 1 (Pass)
    pred_proba = reg_log.predict_proba(user_data)[0][1] 

    # Résultat texte formaté
    result = "✅ PASS (réussi)" if pred_pass == 1 else "❌ FAIL (échoué)"

    # Retourner le score arrondi, la probabilité formatée et le résultat
    return round(pred_score, 2), f"{pred_proba*100:.1f}%", result


# =============================
# 5. INTERFACE GRADIO
# =============================
with gr.Blocks(title="Student Performance Prediction") as demo:
    gr.Markdown("# 🎓 Student Performance Prediction")
    gr.Markdown("---")
    gr.Markdown("Enter the student metrics to predict their Final Score and Pass/Fail Outcome.")

    with gr.Row():
        # Input Sliders
        # The step=0.1 argument here makes the slider accept values with one decimal place.
        study_hours = gr.Slider(minimum=0, maximum=10, value=5, step=0.1, label="Study Hours (0.0 - 10.0)")
        attendance = gr.Slider(minimum=50, maximum=100, value=75, step=1, label="Attendance (%)")
        practice_tests = gr.Slider(minimum=0, maximum=20, value=2, step=1, label="Practice Tests")

    predict_btn = gr.Button("Prédire la Performance")

    with gr.Column():
        # Output components
        final_score = gr.Number(label="📊 Final Score (0-100)", precision=2)
        prob_pass = gr.Textbox(label="🔮 Probabilité de PASS")
        result_text = gr.Textbox(label="🎯 Résultat")

    # Définition de l'action du bouton
    predict_btn.click(fn=predict,
                      inputs=[study_hours, attendance, practice_tests],
                      outputs=[final_score, prob_pass, result_text])
    
    gr.Markdown("---")
    gr.Markdown("## Performance des Modèles (Calculé sur l'ensemble de Test)")

    with gr.Row():
        gr.Textbox(label="Régression Linéaire (Score)", value=f"Erreur Absolue Moyenne (MAE) : {mae:.2f} points", interactive=False, scale=1)
        gr.Textbox(label="Taux d'Erreur Relatif", value=f"Erreur Relative (Score) : {mae_percent:.2f}%", interactive=False, scale=1)

    gr.Textbox(label="Régression Logistique (Pass/Fail)", value=f"Précision (Accuracy) : {accuracy*100:.2f}%", interactive=False)


# Lancer l'app
if __name__ == '__main__':
    # Using share=True is recommended for Colab/Jupyter/remote environments
    demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://1fa7aeb2785835550a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
